# Sprint 4 — Dashboard Integration
## Connecting the HTML Dashboard to the FastAPI Backend
Replaces all hardcoded JS logic in the dashboard with live API calls. After this sprint, every over submission hits the FastAPI backend, and all recommendations, win probability, and alerts come from the ML models.

In [1]:
# This notebook:
# 1. Shows what API calls the dashboard needs to make
# 2. Generates the integration JS layer
# 3. Produces the final connected dashboard HTML

import json
print("Sprint 4 — Dashboard API Integration")
print("Ensure the FastAPI server is running: uvicorn app:app --reload --port 8000")

Sprint 4 — Dashboard API Integration
Ensure the FastAPI server is running: uvicorn app:app --reload --port 8000


## 1. API Integration Map

| Dashboard Action | API Call | Response Used |
|---|---|---|
| Click "Start Match" | `POST /match/setup` | Initialise state |
| Submit over form | `POST /match/over` | score, win%, recs, alerts |
| Page load | `GET /match/state` | Resume in-progress match |
| Right panel refresh | Already in `/match/over` response | bowler_recommendations, alerts |
| WebSocket | `ws://localhost:8000/ws` | Push updates after every over |

In [2]:
# Integration JS layer — replaces the hardcoded logic in the dashboard
JS_INTEGRATION = '''
// ================================================================
// API Integration Layer — replaces hardcoded JS recommendation logic
// Paste this into the dashboard <script> section
// ================================================================

const API_BASE = "http://localhost:8000";
const WS_URL   = "ws://localhost:8000/ws";

let ws = null;

// ── Connect WebSocket ──
function connectWS() {
    ws = new WebSocket(WS_URL);
    ws.onmessage = (evt) => {
        const data = JSON.parse(evt.data);
        applyStateToUI(data);
    };
    ws.onclose = () => setTimeout(connectWS, 2000); // auto-reconnect
    ws.onerror = () => console.warn("WS error — falling back to REST polling");
}

// ── Setup match ──
async function apiSetupMatch(payload) {
    const res = await fetch(`${API_BASE}/match/setup`, {
        method: "POST",
        headers: {"Content-Type": "application/json"},
        body: JSON.stringify(payload)
    });
    if (!res.ok) throw new Error(await res.text());
    return res.json();
}

// ── Submit over ──
async function apiSubmitOver(overNumber, balls, bowler) {
    const res = await fetch(`${API_BASE}/match/over`, {
        method: "POST",
        headers: {"Content-Type": "application/json"},
        body: JSON.stringify({over_number: overNumber, balls, bowler})
    });
    if (!res.ok) throw new Error(await res.text());
    return res.json();
}

// ── Apply API state to dashboard UI ──
function applyStateToUI(state) {
    // Scoreboard
    document.getElementById("score-display").textContent  = state.score;
    document.getElementById("overs-display").textContent  = state.overs + " Ov";
    document.getElementById("crr-display").textContent    = "CRR: " + state.crr;
    document.getElementById("rrr-display").textContent    = state.rrr || "—";
    document.getElementById("projected-display").textContent = state.projected || "—";

    // Win probability
    const wp = state.win_probability;
    document.getElementById("win-prob-fill").style.width = wp + "%";
    document.getElementById("win-prob-pct").textContent  = wp + "%";

    // Batters
    document.getElementById("striker-name").textContent    = state.striker || "—";
    document.getElementById("nonstriker-name").textContent = state.non_striker || "—";

    // Bowler recommendations
    const blist = document.getElementById("bowler-rec-list");
    blist.innerHTML = "";
    (state.bowler_recommendations || []).forEach((b, i) => {
        const card = document.createElement("div");
        card.className = "rec-card" + (i === 0 ? " top-rec" : "");
        card.innerHTML = `
            <div class="rec-card-header">
                <div class="rec-name">${b.name}</div>
                <div class="rec-rank ${i===0?"gold":""}">${i===0?"★ TOP REC":"#"+(i+1)}</div>
            </div>
            <div class="rec-stats-row">
                <div class="rec-stat"><span class="rec-stat-label">Economy</span>
                    <span class="rec-stat-value highlight">${b.economy}</span></div>
                <div class="rec-stat"><span class="rec-stat-label">Wickets</span>
                    <span class="rec-stat-value">${b.wickets}</span></div>
                <div class="rec-stat"><span class="rec-stat-label">Score</span>
                    <span class="rec-stat-value">${b.score}</span></div>
            </div>`;
        blist.appendChild(card);
    });

    // Batter recommendations
    const alist = document.getElementById("batter-rec-list");
    alist.innerHTML = "";
    (state.batter_recommendations || []).forEach((b, i) => {
        const card = document.createElement("div");
        card.className = "rec-card" + (i===0?" top-rec":"");
        card.innerHTML = `
            <div class="rec-card-header">
                <div class="rec-name">${b.name}</div>
                <div class="rec-rank ${i===0?"gold":""}">${i===0?"★ SEND IN":"#2"}</div>
            </div>
            <div class="rec-stats-row">
                <div class="rec-stat"><span class="rec-stat-label">Avg</span>
                    <span class="rec-stat-value highlight">${b.bat_avg}</span></div>
                <div class="rec-stat"><span class="rec-stat-label">SR</span>
                    <span class="rec-stat-value">${b.bat_sr}</span></div>
            </div>`;
        alist.appendChild(card);
    });

    // Alerts
    const abox = document.getElementById("alerts-list");
    abox.innerHTML = "";
    (state.alerts || []).forEach(a => {
        const box = document.createElement("div");
        box.className = `alert-box alert-${a.type}`;
        box.innerHTML = `<div class="alert-title">${a.title}</div>
                         <div class="alert-body">${a.body}</div>`;
        abox.appendChild(box);
    });

    // Over history
    updateOverHistory(state.over_history || []);
}

// ── Modified launchDashboard — calls API ──
async function launchDashboard() {
    const teamAName = document.getElementById("team-a-name").value || "Team A";
    const teamBName = document.getElementById("team-b-name").value || "Team B";
    const tossWon   = document.getElementById("toss-won").value;
    const dec       = document.getElementById("toss-decision").value;

    let battingFirst = "A";
    if ((tossWon === teamBName && dec === "bat") ||
        (tossWon === teamAName && dec === "field")) {
        battingFirst = "B";
    }

    const teamAPlayers = [], teamBPlayers = [];
    for (let i = 0; i < 11; i++) {
        teamAPlayers.push(document.getElementById(`team-a-p${i}`).value);
        teamBPlayers.push(document.getElementById(`team-b-p${i}`).value);
    }

    const payload = {
        team_a: teamAName, team_b: teamBName,
        team_a_players: teamAPlayers,
        team_b_players: teamBPlayers,
        batting_first: battingFirst,
        venue: document.getElementById("venue").value
    };

    try {
        await apiSetupMatch(payload);
        connectWS();
        document.getElementById("setup-screen").style.display = "none";
        document.getElementById("dashboard").style.display    = "flex";
        buildDashboard();
        buildBowlerSelect();
        buildBallInputs();
    } catch (err) {
        alert("API Error: " + err.message + "\nMake sure the FastAPI server is running.");
    }
}

// ── Modified submitOver — calls API ──
async function submitOver() {
    const bowler = document.getElementById("bowler-select").value;
    const balls  = [];
    for (let i = 0; i < 7; i++) {
        const v = document.getElementById(`ball-${i}`).value.trim();
        if (v) balls.push(v);
        else if (i < 6) balls.push("0");
    }
    const overNumber = state.currentOver + 1;
    try {
        const result = await apiSubmitOver(overNumber, balls.slice(0,6), bowler);
        applyStateToUI(result);
        state.currentOver++;
        buildBallInputs();
        updateBattingOrder();
        updateMomentumChart();
        const over = state.currentOver + 1;
        document.getElementById("over-input-title").textContent = `Enter Over ${Math.min(over, 20)}`;
        document.getElementById("balls-remaining-display").textContent =
            `${Math.max(0, (20 - state.currentOver) * 6)} balls left`;
    } catch (err) {
        console.error("API submitOver failed:", err);
        alert("API Error — check server is running.");
    }
}
'''

print("Integration JS layer ready.")
print("Length:", len(JS_INTEGRATION), "chars")

# Save as separate JS file for reference
with open("outputs/dashboard_api_integration.js", "w") as f:
    f.write(JS_INTEGRATION)
print("Saved: outputs/dashboard_api_integration.js")

Integration JS layer ready.
Length: 7312 chars
Saved: outputs/dashboard_api_integration.js


## 2. How to Plug into the Dashboard

In `ipl_dashboard.html`, at the bottom of the `<script>` section, add:

```html
<script src="dashboard_api_integration.js"></script>
```

Or paste the JS directly before the closing `</script>` tag.

The integration replaces three functions:
- `launchDashboard()` → now calls `POST /match/setup`
- `submitOver()` → now calls `POST /match/over`
- `applyStateToUI(state)` → new function that reads API response and updates all panels

In [3]:
# Verify API is reachable and returning correct response schema
import requests, json

BASE_URL = "http://localhost:8000"

def test_endpoint(method, path, payload=None):
    try:
        if method == "GET":
            r = requests.get(f"{BASE_URL}{path}", timeout=3)
        else:
            r = requests.post(f"{BASE_URL}{path}", json=payload, timeout=3)
        print(f"[{r.status_code}] {method} {path}")
        if r.status_code == 200:
            d = r.json()
            print(f"  Keys: {list(d.keys())[:8]}")
        return r.status_code == 200
    except Exception as e:
        print(f"[FAIL] {method} {path} — {e}")
        return False

print("=== API Integration Tests ===\n")
test_endpoint("POST", "/match/setup", {
    "team_a": "MI", "team_b": "CSK",
    "team_a_players": ["Rohit Sharma","Ishan Kishan","Suryakumar Yadav","Hardik Pandya",
                       "Kieron Pollard","Krunal Pandya","Jasprit Bumrah","Trent Boult",
                       "Lasith Malinga","Mitchell McClenaghan","Dwayne Bravo"],
    "team_b_players": ["Ruturaj Gaikwad","Devon Conway","Faf du Plessis","Ambati Rayudu",
                       "MS Dhoni","Ravindra Jadeja","Moeen Ali","Deepak Chahar",
                       "Imran Tahir","Yuzvendra Chahal","Shardul Thakur"],
    "batting_first": "A", "venue": "Wankhede Stadium"
})
test_endpoint("POST", "/match/over", {
    "over_number": 1, "balls": ["0","4","1","0","6","1"], "bowler": "Deepak Chahar"
})
test_endpoint("GET", "/match/state")
test_endpoint("GET", "/players/top-batters")
test_endpoint("GET", "/players/top-bowlers")

=== API Integration Tests ===

[FAIL] POST /match/setup — HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /match/setup (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10751e3d0>: Failed to establish a new connection: [Errno 61] Connection refused'))
[FAIL] POST /match/over — HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /match/over (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x107524650>: Failed to establish a new connection: [Errno 61] Connection refused'))
[FAIL] GET /match/state — HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /match/state (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x1075269d0>: Failed to establish a new connection: [Errno 61] Connection refused'))
[FAIL] GET /players/top-batters — HTTPConnectionPool(host='localhost', port=8000): Max retries exceeded with url: /players/top-ba

False

## 3. CORS & Browser Setup

Since the dashboard is a local HTML file calling the API, the FastAPI CORS middleware is already configured to `allow_origins=["*"]`. 

If opening `ipl_dashboard.html` directly in a browser, you may hit CORS restrictions. Two options:
1. Serve the HTML via uvicorn's static files (recommended)
2. Use the `--disable-web-security` Chrome flag for local dev

**Option 1 — Serve static files (add to `app.py`):**
```python
from fastapi.staticfiles import StaticFiles
app.mount("/", StaticFiles(directory=".", html=True), name="static")
```
Then visit `http://localhost:8000/ipl_dashboard.html`